# Pipeline 3 — Diagnostic Analysis

**Prerequisites**: Run `test_temporal_pipeline.ipynb` cells 1–5 first so that the
following variables are available in memory:

| Variable | Description |
|----------|-------------|
| `df_results` | Per-frame results DataFrame (50 rows) |
| `results` | Raw result list returned by `TemporalSearcher` |
| `aligned` | List of `(csv_idx, ts, frame_path)` tuples |
| `imu_log` | Full IMU/GPS DataFrame with EKF columns |
| `trajectory_gt` | List of `(lat, lon)` ground-truth points |
| `trajectory_est` | List of `(lat, lon)` online-EKF estimates |
| `ekf_errors_online` | Per-frame online EKF errors (m) |
| `map_bounds` | Dict with `lat_min/max`, `lon_min/max` |
| `LOOKAHEAD_M` | Camera look-ahead correction constant |
| `TURN_ROLL_THRESHOLD_RAD`, `TURN_R_MULTIPLIER` | Noise-modulation constants |

Each cell saves a PNG to `../outputs/`.


In [ ]:
# Error CDF + Percentile Comparison Plot
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Left: CDF curves ──
ax = axes[0]
color_map = {
    'Online EKF':  ('blue',   'solid',  2.5,  df_results['online_err'].dropna().values),
    'Batch EKF':   ('red',    'dashed', 1.8,  df_results['batch_err'].dropna().values),
    'Homo raw':    ('green',  'dotted', 1.5,  df_results['homo_err_raw'].dropna().values),
    'Homo corr':   ('purple', 'dashdot',1.5,  df_results['homo_err_corr'].dropna().values),
}
for label, (color, ls, lw, vals) in color_map.items():
    if len(vals) == 0: continue
    xs = np.sort(vals)
    ys = np.linspace(0, 1, len(xs))
    ax.plot(xs, ys * 100, color=color, ls=ls, lw=lw, label=f"{label} (n={len(vals)})")

for thresh, col in [(10,'gray'), (25,'silver'), (50,'gold'), (100,'orange'), (200,'tomato')]:
    ax.axvline(thresh, color=col, ls=':', alpha=0.6, lw=1)
    ax.text(thresh + 1, 2, f'{thresh}m', color=col, fontsize=8, va='bottom')

ax.set_xlabel('Error (m)', fontsize=12)
ax.set_ylabel('Cumulative % of frames', fontsize=12)
ax.set_title('Error CDF — All Methods', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 500)
ax.set_ylim(0, 100)

# ── Right: Error over time (all frames) ──
ax2 = axes[1]
frames = df_results['frame'].values
ax2.plot(frames, df_results['online_err'], 'b-',  lw=1.5, label='Online EKF')
ax2.plot(frames, df_results['batch_err'],  'r--', lw=1.1, alpha=0.6, label='Batch EKF')
hp_frames = df_results[df_results['homo_err_corr'].notna()]['frame'].values
hp_errs   = df_results[df_results['homo_err_corr'].notna()]['homo_err_corr'].values
ax2.scatter(hp_frames, hp_errs, c='purple', s=15, alpha=0.5, zorder=5, label='Homo corr')
gated_frames = df_results[df_results['gate_pass']]['frame'].values
gated_errs   = df_results[df_results['gate_pass']]['online_err'].values
ax2.scatter(gated_frames, gated_errs, c='blue', s=30, alpha=0.35,
            zorder=4, marker='o', edgecolors='navy', linewidths=0.5, label='Gate pass (online)')
ax2.axhline(50,  color='gold',   ls='--', lw=1.5, alpha=0.7, label='50m target')
ax2.axhline(100, color='orange', ls='--', lw=1.0, alpha=0.5, label='100m')

# Shade bank-angle turns
turn_mask = df_results['bank_deg'] > 10
extra_handles = []
if turn_mask.any():
    in_turn = False
    for j, is_turn in enumerate(turn_mask):
        if is_turn and not in_turn:
            turn_start = j; in_turn = True
        elif not is_turn and in_turn:
            ax2.axvspan(turn_start, j, color='orange', alpha=0.12)
            in_turn = False
    if in_turn:
        ax2.axvspan(turn_start, len(turn_mask)-1, color='orange', alpha=0.12)
    extra_handles.append(Patch(facecolor='orange', alpha=0.12, label='Turn (bank>10°)'))

ax2.set_xlabel('Frame', fontsize=12)
ax2.set_ylabel('Error (m)', fontsize=12)
ax2.set_title('Localization Error Over Time (all frames)', fontsize=14)
handles, labels = ax2.get_legend_handles_labels()
ax2.legend(handles=handles + extra_handles, fontsize=9, loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('../outputs/error_cdf_and_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/error_cdf_and_timeline.png")

In [ ]:
# Flight Phase Analysis: Straight vs Turn vs Post-turn
import matplotlib.pyplot as plt
import numpy as np

# Classify each frame by bank angle
# Straight: bank < 5°  |  Turn: bank 5-20°  |  Bank turn: bank > 20°
df = df_results.copy()
df['phase'] = pd.cut(df['bank_deg'], bins=[-1, 5, 20, 90],
                     labels=['straight', 'turn', 'heavy_turn'])

print(f"{'Phase':<14s} | {'N':>4s} | {'Online mean':>11s} | {'Batch mean':>10s} | "
      f"{'Homo corr mean':>14s} | {'Gate%':>6s} | {'<50m':>5s}")
print('-' * 78)
for ph in ['straight', 'turn', 'heavy_turn']:
    sub = df[df['phase'] == ph]
    if sub.empty: continue
    om  = sub['online_err'].mean()
    bm  = sub['batch_err'].mean()
    hm  = sub['homo_err_corr'].dropna().mean()
    gp  = sub['gate_pass'].mean() * 100
    lt50 = (sub['online_err'] < 50).mean() * 100
    print(f"  {ph:<12s} | {len(sub):4d} | {om:>10.1f}m | {bm:>9.1f}m | "
          f"{'N/A' if np.isnan(hm) else f'{hm:>13.1f}m'} | {gp:>5.0f}% | {lt50:>4.0f}%")

# ── Plot error by flight phase ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Error by phase (box plot)
ax = axes[0, 0]
phase_groups = [df[df['phase'] == ph]['online_err'].values
                for ph in ['straight', 'turn', 'heavy_turn']]
labels = ['Straight\n(bank<5°)', 'Turn\n(5-20°)', 'Heavy turn\n(>20°)']
bps = ax.boxplot([g for g, l in zip(phase_groups, labels) if len(g) > 0],
                 labels=[l for g, l in zip(phase_groups, labels) if len(g) > 0],
                 patch_artist=True, notch=False,
                 boxprops=dict(facecolor='lightblue', color='navy'),
                 medianprops=dict(color='red', lw=2))
ax.axhline(50,  color='gold',   ls='--', alpha=0.7, label='50m target')
ax.axhline(100, color='orange', ls='--', alpha=0.5)
ax.set_ylabel('Online EKF Error (m)'); ax.set_title('Error by Flight Phase')
ax.grid(True, alpha=0.3, axis='y'); ax.legend()

# Bank angle and error over time
ax2 = axes[0, 1]
ax2b = ax2.twinx()
ax2.plot(df['frame'], df['online_err'], 'b-', lw=1.5, label='Online err', alpha=0.8)
ax2b.fill_between(df['frame'], df['bank_deg'], alpha=0.25, color='orange', label='Bank angle')
ax2b.set_ylabel('Bank angle (°)', color='orange')
ax2b.tick_params(axis='y', labelcolor='orange')
ax2.set_xlabel('Frame'); ax2.set_ylabel('Error (m)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')
ax2.set_title('Error vs Bank Angle Over Time')
ax2.axhline(50, color='gold', ls='--', alpha=0.5)
ax2.grid(True, alpha=0.2)

# CShape over time
ax3 = axes[1, 0]
ax3.plot(df['frame'], df['CShape'], 'g-o', ms=2.5, lw=1, label='CShape')
ax3.plot(df['frame'], df['inliers'] / df['inliers'].max().clip(1), 'm--', lw=1,
         alpha=0.6, label='Inliers (norm)')
ax3.fill_between(df['frame'], df['bank_deg'] / df['bank_deg'].max().clip(1),
                 alpha=0.15, color='orange', label='Bank (norm)')
ax3.axhline(0.3, color='red', ls='--', alpha=0.5, label='Gate thresh')
ax3.set_xlabel('Frame'); ax3.set_ylabel('Quality metric')
ax3.set_title('Visual Quality + Bank Over Time')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)
ax3.set_ylim(-0.05, 1.05)

# Semantic confidence over time
ax4 = axes[1, 1]
sem_vals = df['sem_conf'].fillna(0.5)
ax4.plot(df['frame'], df['online_err'], 'b-', lw=1.2, alpha=0.7, label='Online err')
sc = ax4.scatter(df['frame'], df['online_err'], c=sem_vals, cmap='RdYlGn',
                 s=20, vmin=0, vmax=1, zorder=5, label='Sem conf (colour)')
plt.colorbar(sc, ax=ax4, label='Semantic confidence', shrink=0.8)
ax4.axhline(50, color='gold', ls='--', alpha=0.5)
ax4.set_xlabel('Frame'); ax4.set_ylabel('Error (m)')
ax4.set_title('Error coloured by Semantic Confidence')
ax4.grid(True, alpha=0.3); ax4.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('../outputs/flight_phase_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/flight_phase_analysis.png")

In [ ]:
# Optimal Look-ahead Calibration (data-driven LOOKAHEAD_M)
import math
import numpy as np
import matplotlib.pyplot as plt

# For each gate-passing frame, measure the raw (uncorrected) offset from homo→GT
# projected onto the heading vector.
raw_forward_offsets = []
for i, (csv_idx, ts, fp) in enumerate(aligned[:len(results)]):
    r = results[i]
    hp_raw = r.get('homo_position')
    if hp_raw is None or not r.get('gate_pass', False):
        continue
    gt_row = imu_log.iloc[csv_idx]
    gt_lat, gt_lon = float(gt_row['gps_lat']), float(gt_row['gps_lon'])
    heading_deg = float(gt_row['yaw_deg']) if 'yaw_deg' in imu_log.columns else 0.0

    # Offset vector in metres (homo_raw → GT)
    d_north = (gt_lat - hp_raw[0]) * 111320.0
    d_east  = (gt_lon - hp_raw[1]) * 111320.0 * math.cos(math.radians(gt_lat))

    # Project onto heading direction
    h = math.radians(heading_deg)
    fwd_n, fwd_e = math.cos(h), math.sin(h)
    forward_component = d_north * fwd_n + d_east * fwd_e
    lateral_component = -d_north * fwd_e + d_east * fwd_n

    raw_forward_offsets.append({
        'frame': i, 'forward': forward_component, 'lateral': lateral_component,
        'dist': math.sqrt(d_north**2 + d_east**2),
        'cs': r.get('visual_quality', {}).get('CShape', 0),
    })

if raw_forward_offsets:
    fwds = np.array([o['forward'] for o in raw_forward_offsets])
    lats = np.array([o['lateral'] for o in raw_forward_offsets])
    print(f"Gate-passing frames with homo position: {len(fwds)}")
    print(f"\nForward component (homo → GT, along heading):")
    print(f"  Mean:   {np.mean(fwds):+8.1f}m  (positive = GT is AHEAD → apply positive lookahead)")
    print(f"  Median: {np.median(fwds):+8.1f}m")
    print(f"  Std:    {np.std(fwds):8.1f}m")
    print(f"\nLateral component (homo → GT, perpendicular to heading):")
    print(f"  Mean:   {np.mean(lats):+8.1f}m  (should be ~0 if no lateral camera tilt)")
    print(f"  Std:    {np.std(lats):8.1f}m")
    opt_lookahead = np.median(fwds)
    print(f"\n{'='*60}")
    print(f"  OPTIMAL LOOKAHEAD_M = {opt_lookahead:.1f}m  (current: {LOOKAHEAD_M}m)")
    print(f"  Applying this would reduce mean forward bias by "
          f"{abs(np.mean(fwds) - opt_lookahead):.1f}m")
    print(f"{'='*60}")

    # Simulate different lookahead values
    test_deltas = np.arange(0, 250, 5)
    residual_mean = []
    residual_median = []
    for delta in test_deltas:
        residuals = np.sqrt((fwds - delta)**2 + lats**2)
        residual_mean.append(np.mean(residuals))
        residual_median.append(np.median(residuals))

    best_mean_idx   = np.argmin(residual_mean)
    best_median_idx = np.argmin(residual_median)
    print(f"\n  Lookahead scan → best mean error:   {test_deltas[best_mean_idx]:.0f}m  "
          f"(residual={residual_mean[best_mean_idx]:.1f}m)")
    print(f"  Lookahead scan → best median error: {test_deltas[best_median_idx]:.0f}m  "
          f"(residual={residual_median[best_median_idx]:.1f}m)")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    ax = axes[0]
    ax.plot(test_deltas, residual_mean,   'b-',  lw=2, label='Mean residual')
    ax.plot(test_deltas, residual_median, 'r--', lw=2, label='Median residual')
    ax.axvline(LOOKAHEAD_M, color='gray', ls=':', lw=2, label=f'Current ({LOOKAHEAD_M}m)')
    ax.axvline(test_deltas[best_mean_idx], color='blue', ls='--', alpha=0.5,
               label=f'Optimal mean ({test_deltas[best_mean_idx]:.0f}m)')
    ax.set_xlabel('Lookahead correction (m)'); ax.set_ylabel('Mean/median residual error (m)')
    ax.set_title('Lookahead Calibration Scan'); ax.legend(); ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    ax2.scatter(fwds, lats, c=df_results[df_results['gate_pass']]['CShape'],
                cmap='RdYlGn', s=30, vmin=0, vmax=1, alpha=0.7)
    ax2.axvline(0, color='gray', ls='-', alpha=0.3)
    ax2.axhline(0, color='gray', ls='-', alpha=0.3)
    ax2.axvline(np.mean(fwds), color='blue', ls='--', lw=2, label=f'Mean fwd={np.mean(fwds):.0f}m')
    circle = plt.Circle((0, 0), 50, color='gold', fill=False, lw=2, ls='--', label='50m radius')
    ax2.add_patch(circle)
    ax2.set_xlabel('Forward offset (m)  [+ve = homo behind drone]')
    ax2.set_ylabel('Lateral offset (m)')
    ax2.set_title('Homo→GT offset in body frame (coloured by CShape)')
    ax2.legend(); ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
    plt.colorbar(ax2.collections[0], ax=ax2, label='CShape')

    plt.tight_layout()
    plt.savefig('../outputs/lookahead_calibration.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/lookahead_calibration.png")
else:
    print("No gate-passing frames available for look-ahead calibration.")

In [ ]:
# Geographic Error Heatmap + Trajectory Quality Map
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# ── Left: Error magnitude heat map on trajectory ──
ax = axes[0]
ax.plot([p[1] for p in trajectory_gt], [p[0] for p in trajectory_gt],
        'g-', lw=0.8, alpha=0.4, label='GT path', zorder=1)

errs = np.array(ekf_errors_online)
errs_clipped = np.clip(errs, 0, 300)
sc = ax.scatter([p[1] for p in trajectory_est], [p[0] for p in trajectory_est],
                c=errs_clipped, cmap='RdYlGn_r', s=18, vmin=0, vmax=300,
                zorder=3, alpha=0.85, label='Online EKF (colour=error)')
plt.colorbar(sc, ax=ax, label='Online EKF error (m, capped 300m)', shrink=0.8)

ax.scatter([trajectory_gt[0][1]], [trajectory_gt[0][0]],
           c='green', s=150, marker='^', zorder=5, edgecolors='k', label='Start')
ax.scatter([trajectory_gt[-1][1]], [trajectory_gt[-1][0]],
           c='red', s=150, marker='v', zorder=5, edgecolors='k', label='End')

ax.axhline(map_bounds['lat_min'], color='gray', ls=':', alpha=0.3)
ax.axhline(map_bounds['lat_max'], color='gray', ls=':', alpha=0.3)
ax.axvline(map_bounds['lon_min'], color='gray', ls=':', alpha=0.3)
ax.axvline(map_bounds['lon_max'], color='gray', ls=':', alpha=0.3)
ax.set_xlabel('Longitude (°E)', fontsize=12)
ax.set_ylabel('Latitude (°N)', fontsize=12)
ax.set_title('Online EKF Error Heatmap on Trajectory', fontsize=14)
ax.legend(fontsize=10, loc='lower left')
ax.grid(True, alpha=0.3); ax.set_aspect('equal')

# ── Right: Gate pass map + homo position cloud ──
ax2 = axes[1]
gated = df_results[df_results['gate_pass']]
ungated = df_results[~df_results['gate_pass']]

ax2.scatter(ungated['online_lon'], ungated['online_lat'],
            c='gray', s=12, alpha=0.4, label='Gate fail (PF/EKF coast)')
ax2.scatter(gated['online_lon'], gated['online_lat'],
            c=gated['online_err'].clip(0, 200), cmap='RdYlGn_r',
            s=25, alpha=0.8, vmin=0, vmax=200, zorder=3,
            label='Gate pass (visual update, colour=err)')
ax2.plot([p[1] for p in trajectory_gt], [p[0] for p in trajectory_gt],
         'g-', lw=1.5, alpha=0.6, label='GT path', zorder=2)

hp_lats = [r['homo_position'][0] for r in results if r.get('homo_position')]
hp_lons = [r['homo_position'][1] for r in results if r.get('homo_position')]
if hp_lats:
    ax2.scatter(hp_lons, hp_lats, c='blue', s=8, alpha=0.2,
                zorder=2, label=f'Homo raw ({len(hp_lats)} frames)', marker='+')

ax2.set_xlabel('Longitude (°E)', fontsize=12)
ax2.set_ylabel('Latitude (°N)', fontsize=12)
ax2.set_title('Gate Pass Map + Visual Measurement Cloud', fontsize=14)
ax2.legend(fontsize=9, loc='lower left')
ax2.grid(True, alpha=0.3); ax2.set_aspect('equal')

plt.tight_layout()
plt.savefig('../outputs/geographic_error_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/geographic_error_map.png")

In [ ]:
# Semantic Confidence & Gate Analysis
import matplotlib.pyplot as plt
import numpy as np

df = df_results.copy()

# ── Semantic confidence bins vs accuracy ──
sc_bins = [0.0, 0.3, 0.5, 0.7, 0.9, 1.01]
sc_labels = ['<0.3', '0.3-0.5', '0.5-0.7', '0.7-0.9', '>0.9']
df['sc_bin'] = pd.cut(df['sem_conf'], bins=sc_bins, labels=sc_labels, right=False)

print("Semantic Confidence vs Accuracy:")
print(f"  {'SC bin':<10s} | {'N':>4s} {'Online err':>10s} | {'Batch err':>9s} | {'Gate%':>6s}")
print(f"  {'-'*55}")
for sc_lbl in sc_labels:
    sub = df[df['sc_bin'] == sc_lbl]
    if sub.empty: continue
    oe = sub['online_err'].mean()
    be = sub['batch_err'].mean()
    gp = sub['gate_pass'].mean() * 100
    print(f"  {sc_lbl:<10s} | {len(sub):4d}  {oe:9.1f}m | {be:8.1f}m | {gp:5.0f}%")

gated   = df[df['gate_pass']]
ungated = df[~df['gate_pass']]
print(f"\nGate PASS ({len(gated)} frames):")
print(f"  Online err: mean={gated['online_err'].mean():.1f}m  median={gated['online_err'].median():.1f}m")
print(f"  Homo corr err (at update): mean={gated['homo_err_corr'].dropna().mean():.1f}m")
print(f"\nGate FAIL ({len(ungated)} frames):")
print(f"  Online err: mean={ungated['online_err'].mean():.1f}m  median={ungated['online_err'].median():.1f}m")
if ungated['homo_err_corr'].notna().any():
    print(f"  Homo corr err (not used): mean={ungated['homo_err_corr'].dropna().mean():.1f}m")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
has_homo = df['homo_err_corr'].notna()
sc = ax.scatter(df[has_homo]['CShape'], df[has_homo]['homo_err_corr'].clip(0, 500),
                c=df[has_homo]['gate_pass'].astype(int), cmap='RdYlGn',
                s=20, alpha=0.7, vmin=0, vmax=1)
ax.axvline(0.3, color='red', ls='--', alpha=0.6, label='Gate thresh')
ax.axvline(0.5, color='orange', ls='--', alpha=0.5, label='High-quality')
ax.axhline(50, color='gold', ls='--', alpha=0.5)
ax.set_xlabel('CShape'); ax.set_ylabel('Homo corr error (m, clip 500)')
ax.set_title('CShape vs Homo Error (green=gate pass)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label='Gate pass')

ax2 = axes[0, 1]
ax2.scatter(df['inliers'].clip(0, 700), df['online_err'].clip(0, 300),
            c=df['gate_pass'].astype(int), cmap='RdYlGn', s=15, alpha=0.5)
ax2.axvline(20,  color='red',    ls='--', alpha=0.6, label='Gate thresh')
ax2.axvline(100, color='orange', ls='--', alpha=0.5)
ax2.axhline(50, color='gold', ls='--', alpha=0.5)
ax2.set_xlabel('Inliers (clip 700)'); ax2.set_ylabel('Online EKF error (m, clip 300)')
ax2.set_title('Inliers vs Online Error')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
ax3.scatter(df['sem_conf'], df['online_err'].clip(0, 300),
            c=df['gate_pass'].astype(int), cmap='RdYlGn', s=15, alpha=0.5)
ax3.set_xlabel('Semantic Confidence'); ax3.set_ylabel('Online EKF error (m, clip 300)')
ax3.set_title('Semantic Confidence vs Online Error')
ax3.grid(True, alpha=0.3); ax3.axhline(50, color='gold', ls='--', alpha=0.5)

ax4 = axes[1, 1]
ax4.hist(df['sem_conf'].dropna(), bins=30, color='steelblue', alpha=0.7, label='All frames')
ax4.hist(df[df['gate_pass']]['sem_conf'].dropna(), bins=30, color='green',
         alpha=0.5, label='Gate pass')
ax4.set_xlabel('Semantic Confidence'); ax4.set_ylabel('Count')
ax4.set_title('Semantic Confidence Distribution')
ax4.legend(); ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/semantic_gate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/semantic_gate_analysis.png")

In [ ]:
# Per-Frame Comparison Table (top 25 worst + best)
import numpy as np

print("TOP 25 WORST ONLINE EKF FRAMES:")
print(f"{'F':>4s} {'time':>6s} {'online':>8s} {'batch':>7s} {'homo_c':>7s} | "
      f"{'CS':>5s} {'inl':>5s} {'sem':>5s} | {'gate':>4s} {'bank':>5s}° | {'method':<18s}")
print('-' * 90)
worst = df_results.nlargest(25, 'online_err')
for _, r in worst.iterrows():
    hc = f"{r['homo_err_corr']:6.0f}m" if not np.isnan(r['homo_err_corr']) else "   N/A"
    print(f"F{r['frame']:3.0f} {r['ts']:6.0f}s | "
          f"{r['online_err']:6.0f}m {r['batch_err']:6.0f}m {hc} | "
          f"{r['CShape']:5.3f} {r['inliers']:5.0f} {r['sem_conf']:5.2f} | "
          f"{'Y' if r['gate_pass'] else 'n':>4s} {r['bank_deg']:5.1f}  | "
          f"{r['method']:<18s}")

print(f"\nTOP 25 BEST ONLINE EKF FRAMES:")
print(f"{'F':>4s} {'time':>6s} {'online':>8s} {'batch':>7s} {'homo_c':>7s} | "
      f"{'CS':>5s} {'inl':>5s} {'sem':>5s} | {'gate':>4s} {'bank':>5s}° | {'method':<18s}")
print('-' * 90)
best = df_results.nsmallest(25, 'online_err')
for _, r in best.iterrows():
    hc = f"{r['homo_err_corr']:6.0f}m" if not np.isnan(r['homo_err_corr']) else "   N/A"
    print(f"F{r['frame']:3.0f} {r['ts']:6.0f}s | "
          f"{r['online_err']:6.0f}m {r['batch_err']:6.0f}m {hc} | "
          f"{r['CShape']:5.3f} {r['inliers']:5.0f} {r['sem_conf']:5.2f} | "
          f"{'Y' if r['gate_pass'] else 'n':>4s} {r['bank_deg']:5.1f}  | "
          f"{r['method']:<18s}")

# Frames where online is WORSE than batch
worse_mask = df_results['online_err'] > df_results['batch_err']
worse_df = df_results[worse_mask].sort_values('online_err', ascending=False)
print(f"\nFrames where ONLINE is WORSE than BATCH ({len(worse_df)}/{len(df_results)}):")
for _, r in worse_df.head(15).iterrows():
    diff = r['online_err'] - r['batch_err']
    print(f"  F{r['frame']:3.0f}: online={r['online_err']:5.0f}m  batch={r['batch_err']:5.0f}m  "
          f"delta={diff:+5.0f}m  CS={r['CShape']:.3f}  bank={r['bank_deg']:.1f}°  "
          f"gate={'Y' if r['gate_pass'] else 'n'}")

In [ ]:
# Timing / Performance Breakdown
import numpy as np
import matplotlib.pyplot as plt

search_times = np.array(df_results['search_time'].values)
print(f"Search time per frame:")
print(f"  Mean:   {search_times.mean():.3f}s")
print(f"  Median: {np.median(search_times):.3f}s")
print(f"  Min:    {search_times.min():.3f}s")
print(f"  Max:    {search_times.max():.3f}s")
print(f"  P90:    {np.percentile(search_times, 90):.3f}s")
print(f"  Total:  {search_times.sum():.1f}s for {len(search_times)} frames")
print(f"  FPS:    {1.0 / search_times.mean():.2f}")

print(f"\nTile search efficiency:")
print(f"  Avg tiles tested:  {df_results['tiles_tested'].mean():.1f}")
print(f"  Max tiles tested:  {df_results['tiles_tested'].max()}")
print(f"  Frames with 0 tiles: {(df_results['tiles_tested'] == 0).sum()} (imu_fallback)")

by_method = df_results.groupby('method')['search_time'].agg(['mean', 'count'])
print(f"\nSearch time by method:")
for m, row in by_method.iterrows():
    print(f"  {m:<22s}: mean={row['mean']:.3f}s  n={int(row['count'])}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(df_results['frame'], df_results['search_time'], 'b-', lw=1.2, alpha=0.8)
ax.axhline(search_times.mean(), color='red', ls='--', label=f'Mean={search_times.mean():.2f}s')
ax.fill_between(df_results['frame'], 0, df_results['search_time'],
                alpha=0.15, color='blue')
ax.set_xlabel('Frame'); ax.set_ylabel('Time (s)'); ax.set_title('Search Time Per Frame')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

ax2 = axes[1]
ax2.hist(search_times, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
ax2.axvline(search_times.mean(), color='red', ls='--', lw=2,
            label=f'Mean={search_times.mean():.2f}s')
ax2.axvline(np.percentile(search_times, 90), color='orange', ls='--', lw=2,
            label=f'P90={np.percentile(search_times, 90):.2f}s')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Count'); ax2.set_title('Time Distribution')
ax2.legend(); ax2.grid(True, alpha=0.3, axis='y')

ax3 = axes[2]
ax3.scatter(df_results['tiles_tested'], df_results['search_time'],
            c=df_results['online_err'].clip(0, 200), cmap='RdYlGn_r',
            s=20, alpha=0.6, vmin=0, vmax=200)
ax3.set_xlabel('Tiles tested'); ax3.set_ylabel('Search time (s)')
ax3.set_title('Tiles Tested vs Search Time')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/timing_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/timing_analysis.png")

In [ ]:
# SP+LG Feature Matching vs Semantic Model: Contribution Analysis
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(22, 12))

# ── 1. CShape vs Semantic Confidence scatter, colored by online error ──
ax = axes[0, 0]
mask_valid = df_results['online_err'].notna()
sc = ax.scatter(df_results.loc[mask_valid, 'CShape'],
                df_results.loc[mask_valid, 'sem_conf'],
                c=df_results.loc[mask_valid, 'online_err'].clip(0, 150),
                cmap='RdYlGn_r', s=25, alpha=0.7, vmin=0, vmax=150)
ax.axvline(0.3, color='red', ls='--', lw=1, alpha=0.6, label='CShape gate (0.3)')
ax.axhline(0.5, color='blue', ls='--', lw=1, alpha=0.6, label='Sem conf 0.5')
ax.set_xlabel('CShape (SP+LG quality)', fontsize=11)
ax.set_ylabel('Semantic Confidence (histogram)', fontsize=11)
ax.set_title('SP+LG Quality vs Semantic Confidence', fontsize=12)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label='Online EKF error (m)', shrink=0.8)

# ── 2. Agreement/disagreement analysis ──
ax = axes[0, 1]
high_sp = df_results['CShape'] > 0.3
high_sem = df_results['sem_conf'] > 0.6
quadrants = {
    'Both High\n(SP+LG + Sem agree)':   high_sp & high_sem,
    'SP+LG High only\n(Sem low)':       high_sp & ~high_sem,
    'Sem High only\n(SP+LG low)':       ~high_sp & high_sem,
    'Both Low\n(both disagree)':         ~high_sp & ~high_sem,
}
colors_q = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']
labels_q = []
means_q = []
counts_q = []
for (lbl, mask), col in zip(quadrants.items(), colors_q):
    errs = df_results.loc[mask, 'online_err']
    labels_q.append(lbl)
    means_q.append(errs.mean() if len(errs) > 0 else 0)
    counts_q.append(len(errs))

bars = ax.bar(range(len(labels_q)), means_q, color=colors_q, alpha=0.8, edgecolor='white')
for i, (b, n) in enumerate(zip(bars, counts_q)):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
            f'n={n}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(range(len(labels_q)))
ax.set_xticklabels(labels_q, fontsize=8)
ax.set_ylabel('Mean Online EKF Error (m)', fontsize=11)
ax.set_title('Agreement Quadrant Analysis', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(50, color='gold', ls='--', lw=1.5, alpha=0.7, label='50m target')
ax.legend(fontsize=8)

# ── 3. SP+LG inliers vs error, colored by semantic confidence ──
ax = axes[0, 2]
gate_mask = df_results['gate_pass']
sc2 = ax.scatter(df_results.loc[gate_mask, 'inliers'],
                 df_results.loc[gate_mask, 'online_err'].clip(0, 200),
                 c=df_results.loc[gate_mask, 'sem_conf'],
                 cmap='RdYlGn', s=25, alpha=0.7, vmin=0.3, vmax=1.0)
ax.set_xlabel('SP+LG Inliers', fontsize=11)
ax.set_ylabel('Online EKF Error (m)', fontsize=11)
ax.set_title('Inliers vs Error (gate-pass only)', fontsize=12)
plt.colorbar(sc2, ax=ax, label='Semantic Confidence', shrink=0.8)
ax.axhline(50, color='gold', ls='--', lw=1.5, alpha=0.7)
ax.grid(True, alpha=0.3)

# ── 4. Time series: SP+LG CShape and Semantic Confidence (overlaid) ──
ax = axes[1, 0]
ax.plot(df_results['frame'], df_results['CShape'], 'b-', lw=1.2, alpha=0.8, label='CShape (SP+LG)')
ax.plot(df_results['frame'], df_results['sem_conf'], 'g-', lw=1.2, alpha=0.8, label='Sem Conf')
ax.axhline(0.3, color='red', ls=':', lw=1, alpha=0.5, label='CShape gate')
fail_mask = ~df_results['gate_pass']
for j, fail in enumerate(fail_mask):
    if fail:
        ax.axvspan(j-0.5, j+0.5, color='red', alpha=0.05)
ax.set_xlabel('Frame', fontsize=11)
ax.set_ylabel('Confidence Score', fontsize=11)
ax.set_title('SP+LG vs Semantic Confidence Over Time', fontsize=12)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# ── 5. Measurement noise R modulation effect ──
ax = axes[1, 1]
R_base = np.where(
    (df_results['CShape'] > 0.5) & (df_results['inliers'] > 100),
    30.0**2, 60.0**2)
sem_factor = np.maximum(0.5, 2.0 - 1.5 * df_results['sem_conf'])
R_effective = R_base * sem_factor
turn_factor = np.where(df_results['bank_deg'] > np.degrees(TURN_ROLL_THRESHOLD_RAD),
                       TURN_R_MULTIPLIER, 1.0)
R_effective *= turn_factor
R_sqrt = np.sqrt(R_effective)
ax.plot(df_results['frame'], R_sqrt, 'purple', lw=1.2, alpha=0.8, label='√R effective')
ax.fill_between(df_results['frame'], 0, R_sqrt, alpha=0.15, color='purple')
ax2b = ax.twinx()
ax2b.plot(df_results['frame'], df_results['online_err'].clip(0, 200),
          'b-', lw=0.8, alpha=0.5, label='Online error')
ax2b.set_ylabel('Error (m)', fontsize=11, color='blue')
ax.set_xlabel('Frame', fontsize=11)
ax.set_ylabel('√R measurement noise (m)', fontsize=11, color='purple')
ax.set_title('Adaptive Measurement Noise Modulation', fontsize=12)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)

# ── 6. Feature importance: Spearman correlation ──
ax = axes[1, 2]
gated_df = df_results[df_results['gate_pass']].copy()
if len(gated_df) > 10:
    from scipy import stats
    corr_sp, p_sp = stats.spearmanr(gated_df['CShape'], gated_df['online_err'])
    corr_sem, p_sem = stats.spearmanr(gated_df['sem_conf'], gated_df['online_err'])
    corr_inl, p_inl = stats.spearmanr(gated_df['inliers'], gated_df['online_err'])

    feats = ['CShape\n(SP+LG)', 'Inliers\n(SP+LG)', 'Sem Conf\n(Semantic)']
    corrs = [corr_sp, corr_inl, corr_sem]
    colors_c = ['#3498db', '#2980b9', '#27ae60']
    bars2 = ax.barh(range(len(feats)), corrs, color=colors_c, alpha=0.8, edgecolor='white')
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels(feats, fontsize=10)
    ax.set_xlabel('Spearman Correlation with Error\n(negative = higher quality → lower error)', fontsize=10)
    ax.set_title('Feature Importance: Who Drives Accuracy?', fontsize=12)
    for i, (b, c, p) in enumerate(zip(bars2, corrs, [p_sp, p_inl, p_sem])):
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ax.text(c + 0.01 if c > 0 else c - 0.01, i,
                f'rho={c:.3f} {sig}', va='center',
                ha='left' if c > 0 else 'right', fontsize=9)
    ax.axvline(0, color='black', lw=0.8)
    ax.grid(True, alpha=0.3, axis='x')
    ax.set_xlim(-0.5, 0.3)
else:
    ax.text(0.5, 0.5, 'Insufficient gate-pass\nframes for analysis',
            ha='center', va='center', transform=ax.transAxes, fontsize=14)

plt.suptitle('SuperPoint+LightGlue vs Semantic Model — Contribution Analysis',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/splg_vs_semantic_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print("\n" + "="*78)
print("  SP+LG vs SEMANTIC MODEL — CONTRIBUTION SUMMARY")
print("="*78)
for lbl, mask in quadrants.items():
    n = mask.sum()
    if n == 0: continue
    errs = df_results.loc[mask, 'online_err']
    lt50 = (errs < 50).mean() * 100
    print(f"  {lbl.replace(chr(10), ' '):<35s}: n={n:3d}  mean={errs.mean():6.1f}m  "
          f"<50m={lt50:4.0f}%")

print(f"\n  SP+LG is the PRIMARY position estimator (homography -> lat/lon).")
print(f"  Semantic model SUPPORTS by:")
print(f"    1. Pre-filtering tiles (reduces ~24 candidates -> top-10 before SP+LG)")
print(f"    2. Modulating measurement noise R (high sem_conf -> trust more)")
print(f"    3. Double-confirmation (logs terrain match for diagnostics)")
if len(gated_df) > 10:
    print(f"\n  Correlation with error (gate-pass frames):")
    print(f"    CShape (SP+LG):     rho={corr_sp:+.3f}  {'(significant)' if p_sp < 0.05 else '(not significant)'}")
    print(f"    Inliers (SP+LG):    rho={corr_inl:+.3f}  {'(significant)' if p_inl < 0.05 else '(not significant)'}")
    print(f"    Sem Conf (Semantic): rho={corr_sem:+.3f}  {'(significant)' if p_sem < 0.05 else '(not significant)'}")
print("Saved: outputs/splg_vs_semantic_comparison.png")